### Topic: Practical Implementation of Tools in LangGraph

In [3]:
#### Step 1: Import Necessary Libraries

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition

# Build in Tools -> DuckDuckGoSearchRun
from langchain_community.tools import DuckDuckGoSearchRun
# To build the custom tool
from langchain_core.tools import tool

import requests
import random



c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
C:\Users\kz\AppData\Local\Temp\ipykernel_1244\3848739813.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
load_dotenv()

True

### 1. Define build-in tools

#### 1. Tool Name: DuckDuckGoSearchRun

- Purpose: To search on internet

- use process:
- install:
    - pip install -U duckduckgo-search
    - pip install -U duckduckgo-search langchain-community
    - pip install -U ddgs

In [ ]:
# DuckDuckGoSearchRun

search_tool_1 = DuckDuckGoSearchRun(region="us-en") 

In [1]:
# ============================
# Tavily Search
# =============================
from langchain_tavily import TavilySearch

ModuleNotFoundError: No module named 'langchain_tavily'

In [ ]:
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
search_tool_2 = TavilySearch(max_results=4)  

### 2. Define the Custom Tool

In [ ]:
# 2.1: create a calculator tool

@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}


In [ ]:
# 2.2: Create get_stock_price tool
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    r = requests.get(url)
    return r.json()

In [ ]:
# Make tool list
tools = [search_tool_1, search_tool_2, get_stock_price, calculator]

# Make the LLM tool-aware
llm_with_tools = llm.bind_tools(tools)

### Create a State

In [4]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
# graph nodes
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


In [ ]:

tool_node = ToolNode(tools)  # Executes tool calls

In [ ]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [ ]:
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

# always go to tools node to chat_note(where the llm is present)
graph.add_edge("tools", "chat_node")    

In [ ]:
chatbot = graph.compile()

chatbot

In [ ]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)


In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple")]})
print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple using get stock price tool then use the calculator tool to find out how much will it take to purchase 50 shares?")]})
print(out["messages"][-1].content)